# LMI — ذكاء سوق العمل اليمني · تشغيل كامل من Google Drive
**قبل البدء:** Runtime → Change runtime type → **T4 GPU**.

**مرة واحدة فقط:** ارفع `LMI_PROJECT.zip` إلى جذر **My Drive** (سحب وإفلات من المتصفح).

ما يفعله الدفتر:
1. يربط Drive ويفك المشروع إلى `My Drive/LMI_PROJECT` (مرة واحدة).
2. يدرّب نماذج AraBERT الثلاثة **ويحفظها في Drive مباشرة** — ويتخطى أي نموذج مدرب مسبقاً (إن انقطعت الجلسة أعد التشغيل فيكمل).
3. يقيّم على test ويولّد `docs/RESULTS.md`.
4. يشغّل الاختبارات ثم **التطبيق مربوطاً بالنماذج** برابط عام.

الأجزاء المنفذة مسبقاً على البيانات الكاملة (التدقيق، خطوط الأساس، LSTM) موجودة في الحزمة ولا تُعاد.

## 1) ربط Drive وتجهيز المشروع

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, shutil, subprocess, json, sys, time
MY = "/content/drive/MyDrive"
PROJ = f"{MY}/LMI_PROJECT"                       # المشروع الدائم في Drive (النماذج والتقارير تُحفظ هنا)
if not os.path.exists(f"{PROJ}/src/common.py"):
    z = f"{MY}/LMI_PROJECT.zip"
    assert os.path.exists(z), "ارفع LMI_PROJECT.zip إلى جذر My Drive أولاً"
    print("فك الضغط إلى Drive…"); zipfile.ZipFile(z).extractall(MY)
# نسخة عمل محلية سريعة للكود والبيانات (القراءة من Drive بطيئة أثناء التدريب)
ROOT = "/content/lmi"
if not os.path.exists(f"{ROOT}/src/common.py"):
    shutil.copytree(PROJ, ROOT, ignore=shutil.ignore_patterns("models_dl", "reports"))
os.chdir(ROOT); sys.path.insert(0, ROOT)
os.environ["LMI_FINAL_DIR"]   = f"{ROOT}/FINAL_TRAINING_DATASET"
os.environ["LMI_MODELS_DIR"]  = f"{PROJ}/models_dl"     # ← النماذج في Drive
os.environ["LMI_REPORTS_DIR"] = f"{PROJ}/reports"       # ← التقارير في Drive
print("المشروع:", PROJ); print(sorted(os.listdir(PROJ)))

In [ ]:
!pip -q install "transformers>=4.44" seqeval sklearn-crfsuite arabert accelerate safetensors streamlit plotly
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠ لا يوجد GPU — التدريب سيكون بطيئاً جداً")

## 2) تدريب AraBERT (يتخطى المدرب مسبقاً — آمن لإعادة التشغيل)
أفضل Checkpoint على validation فقط؛ test لا يُقرأ أثناء التدريب.

In [ ]:
M = os.environ["LMI_MODELS_DIR"]
def trained(d): return os.path.exists(f"{d}/config.json") and os.path.exists(f"{d}/training_config.json")
for task in ["opportunity_type", "geographic_scope"]:
    d = f"{M}/classification/{task}"
    if trained(d): print(f"✔ {task}: مدرب مسبقاً في Drive — تخطٍ"); continue
    t = time.time()
    !python -m src.training.train_classifier --task {task}
    print(f"⏱ {task}: {(time.time()-t)/60:.1f} دقيقة")

In [ ]:
d = f"{M}/ner"
if trained(d): print("✔ NER: مدرب مسبقاً في Drive — تخطٍ")
else:
    t = time.time()
    !python -m src.training.train_ner
    print(f"⏱ NER: {(time.time()-t)/60:.1f} دقيقة")

## 3) التقييم المستقل على test + المقارنة بخطوط الأساس

In [ ]:
R = os.environ["LMI_REPORTS_DIR"]
for task in ["opportunity_type", "geographic_scope"]:
    if not os.path.exists(f"{R}/evaluation/classification_{task}/metrics.json"):
        !python -m src.evaluation.evaluate_classifier --task {task}
if not os.path.exists(f"{R}/evaluation/ner/metrics.json"):
    !python -m src.evaluation.evaluate_ner
!python scripts/build_results_section.py
shutil.copy(f"{ROOT}/docs/RESULTS.md", f"{PROJ}/docs/RESULTS.md")
from IPython.display import Markdown, display
display(Markdown(open(f"{PROJ}/docs/RESULTS.md", encoding="utf-8").read()))

## 4) الاختبارات

In [ ]:
!python -m pytest -q tests/test_dl_system.py -p no:cacheprovider 2>&1 | tail -3

## 5) تشغيل التطبيق مربوطاً بالنماذج
يقرأ البيانات من النسخة المحلية والنماذج والتقارير من Drive. يظهر رابط `trycloudflare.com` — افتحه. أوقف الخلية لإيقاف التطبيق.

In [ ]:
import re, subprocess, time
if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
app = subprocess.Popen(["streamlit", "run", "app_dl/main.py", "--server.port", "8501", "--server.headless", "true"],
                       cwd=ROOT, env=os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:
    m = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if m: print("🌐 رابط التطبيق:", m.group(0)); break
try:
    while True: time.sleep(60)
except KeyboardInterrupt:
    app.terminate(); tun.terminate(); print("أُوقف التطبيق")

## 6) (اختياري) تشغيل التطبيق على الكمبيوتر
ينشئ `lmi_models.zip` (أوزان fp16) و`lmi_reports.zip` في Drive (≈ نصف الحجم). نزّله من Drive وفكّه داخل مجلد المشروع على جهازك ثم:
```
pip install -r requirements.txt -r requirements_dl.txt
streamlit run app_dl/main.py
```

In [ ]:
!python scripts/package_results.py --out {MY} --fp16
!ls -la {MY}/lmi_models.zip {MY}/lmi_reports.zip